# Phase 4 — LangGraph Agentic RAG Workflow

**Project:** Enterprise Customer Support Automation  
**Goal:** Turn the Phase 2 hybrid retriever and Phase 3 FLAN-T5 answer generator into a small, inspectable LangGraph workflow.

### What this phase adds
1. **Planner/router node** — decides how to handle the question.
2. **Retrieval node** — searches the existing FAISS + BM25 hybrid knowledge base.
3. **Answer node** — generates an answer grounded in retrieved chunks.
4. **Validation node** — checks whether evidence was found and whether the answer is usable.
5. **One retry** — if evidence is weak or the answer is empty, rewrite the query and retrieve again.
6. **Lightweight memory** — optionally includes a small amount of recent conversation history.
7. **Evaluation outputs** — save agentic answers and workflow traces for comparison with Phase 3.

This notebook is designed for a beginner and uses your local `google/flan-t5-small` model; no paid API key is required. It reuses the Phase 1 and Phase 2 files. It does not yet create the FastAPI service or Streamlit UI; those belong in later phases.

## Cell 1 — Locate the project

Save this notebook as `notebooks/04_phase4_langgraph_agentic_rag.ipynb` in the same project folder as the earlier notebooks. This cell searches upward from the current working directory for the Phase 1 chunk file.

In [1]:
from pathlib import Path
import sys, json, re, os, time

cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "data" / "processed" / "knowledge_base_chunks.jsonl").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot find data/processed/knowledge_base_chunks.jsonl. "
        "Save this notebook in your project's notebooks folder and confirm Phase 1 completed."
    )

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

Project root: C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents
Python: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\.venv\Scripts\python.exe


## Cell 2 — Install LangGraph

Run once. This installs LangGraph without forcing an upgrade of Transformers, which caused a compatibility issue in Phase 3. If the kernel asks to restart, restart it and run the notebook from Cell 1 again.

In [2]:
%pip install langgraph

  Using cached pydantic-2.14.1-py3-none-any.whl.metadata (133 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.50.1-cp314-cp314-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached charset_normalizer-3.5.2-cp314-cp314-win_amd64.whl.metadata (47 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
   ---------------------------------------- 0.0/573.3 kB ? eta -:--:--
   ------------------------------------ --- 524.3/573.3 kB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 573.3/573.3 kB 1.4 MB/s  0:00:00
   ---------------------------------------- 0.0/862.4 kB ? eta -:--:--
   ---------------------------------------- 862.4/862.4 kB 5.7 MB/s  0:00:00
Using cached pydantic-2.14.1-py3-none-any.whl (488 kB)
Using cached pydantic_core-2.50.1-cp314-cp314-win_amd64.whl (2.2 

## Cell 3 — Import libraries

In [3]:
import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from typing import TypedDict, List, Dict, Any, Optional
from IPython.display import display
from langgraph.graph import StateGraph, END

print("Imports successful.")
print("PyTorch device:", "CUDA GPU" if torch.cuda.is_available() else "CPU")

c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful.
PyTorch device: CPU


## Cell 4 — Load the Phase 1 chunks and Phase 2 FAISS index

Do not regenerate these artifacts here. Phase 4 reuses them.

In [4]:
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "knowledge_base_chunks.jsonl"
FAISS_PATH = PROJECT_ROOT / "data" / "vector_store" / "customer_support.faiss"

chunks = []
with CHUNKS_PATH.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if line.strip():
            try:
                chunks.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on line {line_no}: {exc}") from exc

chunks_df = pd.DataFrame(chunks)
if chunks_df.empty:
    raise ValueError("No knowledge-base chunks were loaded.")

TEXT_CANDIDATES = ["chunk_text", "text", "content", "page_content", "cleaned_text"]
TEXT_FIELD = next((name for name in TEXT_CANDIDATES if name in chunks_df.columns), None)
if TEXT_FIELD is None:
    raise KeyError(f"No text column found. Available columns: {list(chunks_df.columns)}")

ID_CANDIDATES = ["chunk_id", "id", "chunkId"]
ID_FIELD = next((name for name in ID_CANDIDATES if name in chunks_df.columns), None)
if ID_FIELD is None:
    chunks_df["generated_chunk_id"] = [f"chunk_{i:04d}" for i in range(len(chunks_df))]
    ID_FIELD = "generated_chunk_id"

chunks_df[TEXT_FIELD] = chunks_df[TEXT_FIELD].fillna("").astype(str).str.strip()
chunks_df = chunks_df[chunks_df[TEXT_FIELD].ne("")].reset_index(drop=True)
texts = chunks_df[TEXT_FIELD].tolist()

if not FAISS_PATH.exists():
    raise FileNotFoundError(f"Missing FAISS index: {FAISS_PATH}. Complete Phase 2 first.")
faiss_index = faiss.read_index(str(FAISS_PATH))

if faiss_index.ntotal != len(chunks_df):
    raise ValueError(
        f"FAISS has {faiss_index.ntotal} vectors but {len(chunks_df)} chunks were loaded. "
        "Check that the Phase 1 and Phase 2 artifacts belong to the same dataset."
    )

print("Chunks loaded:", len(chunks_df))
print("FAISS vectors:", faiss_index.ntotal)
print("Text field:", TEXT_FIELD)
print("ID field:", ID_FIELD)
display(chunks_df.head(3))

Chunks loaded: 18
FAISS vectors: 18
Text field: text
ID field: chunk_id


,chunk_id,document_id,title,category,source,chunk_index,text,word_count
0,FAQ001_CHUNK001,FAQ001,Track an order,orders,sample_faq,1,Customers can check an order's status from the...,45
1,FAQ002_CHUNK001,FAQ002,Cancel an order,orders,sample_faq,1,An order can usually be cancelled while it is ...,49
2,POL001_CHUNK001,POL001,Shipping and delivery policy,shipping,sample_policy,1,Standard delivery typically takes 3 to 7 busin...,51


## Cell 5 — Load the embedding model and build keyword search

In [5]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

def tokenize(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

bm25_index = BM25Okapi([tokenize(text) for text in texts])
print("Embedding model ready:", EMBEDDING_MODEL_NAME)
print("BM25 keyword index ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2205.87it/s]


Embedding model ready: sentence-transformers/all-MiniLM-L6-v2
BM25 keyword index ready.


## Cell 6 — Define hybrid retrieval

Semantic search finds conceptually similar chunks. BM25 keyword search helps match exact terms. Reciprocal Rank Fusion combines their rankings.

In [6]:
def semantic_search(query, top_k=5):
    vector = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    k = min(max(1, int(top_k)), len(texts))
    scores, indices = faiss_index.search(vector, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if 0 <= int(idx) < len(chunks_df):
            row = chunks_df.iloc[int(idx)].to_dict()
            row["semantic_score"] = float(score)
            results.append(row)
    return results

def keyword_search(query, top_k=5):
    scores = bm25_index.get_scores(tokenize(query))
    ranked = np.argsort(scores)[::-1][:min(max(1, int(top_k)), len(texts))]
    results = []
    for idx in ranked:
        if float(scores[int(idx)]) > 0:
            row = chunks_df.iloc[int(idx)].to_dict()
            row["bm25_score"] = float(scores[int(idx)])
            results.append(row)
    return results

def hybrid_search(query, top_k=4, candidate_k=10, rrf_k=60):
    semantic = semantic_search(query, min(candidate_k, len(texts)))
    keyword = keyword_search(query, min(candidate_k, len(texts)))
    fused, by_id = {}, {}
    for result_list in (semantic, keyword):
        for rank, item in enumerate(result_list, start=1):
            key = str(item[ID_FIELD])
            fused[key] = fused.get(key, 0.0) + 1.0 / (rrf_k + rank)
            by_id[key] = item
    ranked_ids = sorted(fused, key=fused.get, reverse=True)[:min(max(1, int(top_k)), len(fused))]
    results = []
    for rank, key in enumerate(ranked_ids, start=1):
        item = dict(by_id[key])
        item["rrf_score"] = float(fused[key])
        item["hybrid_rank"] = rank
        results.append(item)
    return results

print("Hybrid retrieval ready.")

Hybrid retrieval ready.


## Cell 7 — Load FLAN-T5 without the removed pipeline task

This follows the compatible direct-loading approach used in the corrected Phase 3 notebook. It avoids `pipeline("text2text-generation", ...)`, which is removed in Transformers v5.

In [7]:
GENERATION_MODEL_NAME = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL_NAME)
generation_model = AutoModelForSeq2SeqLM.from_pretrained(GENERATION_MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generation_model = generation_model.to(device)
generation_model.eval()

def generator(prompt, max_new_tokens=160, do_sample=False):
    inputs = tokenizer(
        str(prompt), return_tensors="pt", truncation=True, max_length=512
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        output_ids = generation_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=do_sample
        )
    answer_text = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    return answer_text

print("Generation model ready:", GENERATION_MODEL_NAME)
print("Device:", device)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 303.19it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generation model ready: google/flan-t5-small
Device: cpu


## Cell 8 — Build a grounded prompt

In [8]:
def build_rag_prompt(question, retrieved_chunks, conversation_history=None, max_context_chars=3500):
    evidence_blocks = []
    used_chars = 0
    for item in retrieved_chunks:
        chunk_text = str(item[TEXT_FIELD]).strip()
        chunk_id = str(item[ID_FIELD])
        title = str(item.get("title", "Untitled"))
        block = f"[Source ID: {chunk_id} | Title: {title}]\n{chunk_text}\n"
        if used_chars + len(block) > max_context_chars:
            remaining = max_context_chars - used_chars
            if remaining > 100:
                evidence_blocks.append(block[:remaining])
            break
        evidence_blocks.append(block)
        used_chars += len(block)

    context = "\n".join(evidence_blocks) if evidence_blocks else "(No relevant evidence found.)"
    history_text = ""
    if conversation_history:
        recent = conversation_history[-3:]
        history_text = "\nRECENT CONVERSATION (use only to resolve references):\n" + "\n".join(
            f"User: {item.get('question', '')}\nAssistant: {item.get('answer', '')}" for item in recent
        )
    return (
        "You are a helpful enterprise customer support assistant. Answer using only the support context below. "
        "Do not invent policies, dates, fees, or details. If the context does not answer the question, clearly say "
        "you cannot confirm it from the available support documents. Be concise and cite source IDs in square brackets.\n\n"
        f"CONTEXT:\n{context}{history_text}\n\nQUESTION: {question}\nANSWER:"
    )

print("Grounded prompt builder ready.")

Grounded prompt builder ready.


## Cell 9 — Define the LangGraph state

LangGraph passes a shared state dictionary between nodes. These fields track the question, retrieval results, answer, validation, retry count, and a trace that helps us explain the workflow in the report.

In [9]:
class AgentState(TypedDict, total=False):
    question: str
    rewritten_question: str
    active_query: str
    conversation_history: List[Dict[str, str]]
    retrieved_chunks: List[Dict[str, Any]]
    answer: str
    sources: List[Dict[str, str]]
    evidence_found: bool
    answer_valid: bool
    retry_count: int
    max_retries: int
    trace: List[str]
    status: str

## Cell 10 — Create the planner, retrieval, answer, and validation nodes

In [10]:
def planner_node(state: AgentState) -> Dict[str, Any]:
    question = str(state.get("question", "")).strip()
    if not question:
        return {
            "active_query": "",
            "trace": state.get("trace", []) + ["planner: empty question"],
            "status": "empty_question"
        }

    # Lightweight planning: preserve the question, then route it to hybrid retrieval.
    # A production planner could classify intent or use an LLM, but this version is deterministic.
    active_query = question
    return {
        "active_query": active_query,
        "retry_count": int(state.get("retry_count", 0)),
        "max_retries": int(state.get("max_retries", 1)),
        "trace": state.get("trace", []) + ["planner: selected hybrid retrieval"]
    }

def retrieval_node(state: AgentState) -> Dict[str, Any]:
    query = state.get("active_query") or state.get("question", "")
    retrieved = hybrid_search(query, top_k=4, candidate_k=10)
    evidence_found = len(retrieved) > 0
    trace_note = f"retrieval: found {len(retrieved)} chunk(s) for query={query!r}"
    return {
        "retrieved_chunks": retrieved,
        "evidence_found": evidence_found,
        "trace": state.get("trace", []) + [trace_note]
    }

def answer_node(state: AgentState) -> Dict[str, Any]:
    question = str(state.get("question", "")).strip()
    retrieved = state.get("retrieved_chunks", [])
    if not retrieved:
        answer = "I cannot confirm the answer from the available support documents."
    else:
        prompt = build_rag_prompt(
            question,
            retrieved,
            conversation_history=state.get("conversation_history", [])
        )
        answer = generator(prompt, max_new_tokens=160, do_sample=False)

    sources = [{
        "chunk_id": str(item[ID_FIELD]),
        "title": str(item.get("title", "Untitled")),
        "category": str(item.get("category", "")),
        "text": str(item[TEXT_FIELD])
    } for item in retrieved]

    return {
        "answer": answer,
        "sources": sources,
        "trace": state.get("trace", []) + ["answer: generated grounded response"]
    }

def validation_node(state: AgentState) -> Dict[str, Any]:
    answer = str(state.get("answer", "")).strip()
    retrieved = state.get("retrieved_chunks", [])
    # This is a simple operational validation, not a guarantee of factual correctness.
    # For this workflow, a response is considered operationally valid only if
    # it is non-empty and has retrieved evidence. Missing evidence triggers one retry.
    answer_valid = bool(answer) and bool(retrieved)
    trace_note = "validation: passed" if answer_valid else "validation: needs retry"
    return {
        "answer_valid": answer_valid,
        "trace": state.get("trace", []) + [trace_note],
        "status": "complete" if answer_valid else "retry_needed"
    }

def rewrite_query_node(state: AgentState) -> Dict[str, Any]:
    original = str(state.get("question", "")).strip()
    # Simple rewrite expands the query with support-related terms.
    rewritten = f"{original} customer support policy details"
    retry_count = int(state.get("retry_count", 0)) + 1
    return {
        "rewritten_question": rewritten,
        "active_query": rewritten,
        "retry_count": retry_count,
        "trace": state.get("trace", []) + [f"query rewrite: {rewritten!r}"]
    }

def route_after_validation(state: AgentState) -> str:
    if state.get("answer_valid", False):
        return "end"
    if int(state.get("retry_count", 0)) < int(state.get("max_retries", 1)):
        return "retry"
    return "end"

print("Agent nodes defined.")

Agent nodes defined.


## Cell 11 — Connect the nodes into a LangGraph workflow

The graph runs planner → retrieval → answer → validation. If validation fails, it rewrites the query and retries retrieval/answer once, then ends.

In [11]:
workflow = StateGraph(AgentState)
workflow.add_node("planner", planner_node)
workflow.add_node("retrieve", retrieval_node)
workflow.add_node("answer", answer_node)
workflow.add_node("validate", validation_node)
workflow.add_node("rewrite_query", rewrite_query_node)

workflow.set_entry_point("planner")
workflow.add_edge("planner", "retrieve")
workflow.add_edge("retrieve", "answer")
workflow.add_edge("answer", "validate")
workflow.add_conditional_edges(
    "validate",
    route_after_validation,
    {
        "retry": "rewrite_query",
        "end": END
    }
)
workflow.add_edge("rewrite_query", "retrieve")

agentic_rag = workflow.compile()
print("LangGraph workflow compiled successfully.")

LangGraph workflow compiled successfully.


## Cell 12 — Run one end-to-end question

Edit `question` to try another customer support scenario. `conversation_history` is optional; keep it empty for a standalone question.

In [12]:
question = "How long do I have to return an item?"

initial_state = {
    "question": question,
    "conversation_history": [],
    "retry_count": 0,
    "max_retries": 1,
    "trace": []
}

result = agentic_rag.invoke(initial_state)

print("QUESTION:", result.get("question"))
print("\nANSWER:\n", result.get("answer"))
print("\nSTATUS:", result.get("status"))
print("\nSOURCES:")
for source in result.get("sources", []):
    print(f"- {source['chunk_id']} | {source['title']}")
print("\nWORKFLOW TRACE:")
for step in result.get("trace", []):
    print("-", step)

QUESTION: How long do I have to return an item?

ANSWER:
 30 calendar days

STATUS: complete

SOURCES:
- POL002_CHUNK001 | Returns policy
- FAQ002_CHUNK001 | Cancel an order
- FAQ004_CHUNK001 | Request a refund
- FAQ005_CHUNK001 | Item arrived damaged

WORKFLOW TRACE:
- planner: selected hybrid retrieval
- retrieval: found 4 chunk(s) for query='How long do I have to return an item?'
- answer: generated grounded response
- validation: passed


## Cell 13 — Inspect the retrieved evidence

In [13]:
for rank, item in enumerate(result.get("retrieved_chunks", []), start=1):
    print("=" * 80)
    print(f"Rank {rank} | Chunk ID: {item[ID_FIELD]} | Title: {item.get('title', 'Untitled')}")
    print(str(item[TEXT_FIELD])[:1000])

Rank 1 | Chunk ID: POL002_CHUNK001 | Title: Returns policy
Eligible items may be returned within 30 calendar days of delivery. Items should be unused and returned with their original packaging, unless they arrived damaged or defective. Some final-sale or customized items may not qualify. Customers should start a return from the order details page and follow the instructions provided. The return request must be approved before an item is sent back.
Rank 2 | Chunk ID: FAQ002_CHUNK001 | Title: Cancel an order
An order can usually be cancelled while it is still in the Processing status. Open Orders, select the order, and choose Cancel Order if the option is available. Orders that have already shipped cannot normally be cancelled; customers may request a return after delivery according to the returns policy.
Rank 3 | Chunk ID: FAQ004_CHUNK001 | Title: Request a refund
After a return is received and inspected, an approved refund is sent to the original payment method. The time it takes for t

## Cell 14 — Run a small agentic evaluation

We reuse the five Phase 3 evaluation questions so you can compare baseline and agentic answers. This small test checks execution and creates review material; it is not a statistically robust evaluation.

In [14]:
evaluation_data = [
    {"question": "How can I track my order?"},
    {"question": "What should I do if my item arrives damaged?"},
    {"question": "How long do I have to return an item?"},
    {"question": "Which payment methods are accepted?"},
    {"question": "How can I keep my account secure?"}
]

agentic_rows = []
for i, item in enumerate(evaluation_data, start=1):
    print(f"Running agentic question {i}/{len(evaluation_data)}...")
    state = agentic_rag.invoke({
        "question": item["question"],
        "conversation_history": [],
        "retry_count": 0,
        "max_retries": 1,
        "trace": []
    })
    source_ids = [str(x["chunk_id"]) for x in state.get("sources", [])]
    agentic_rows.append({
        "question": item["question"],
        "agentic_answer": state.get("answer", ""),
        "source_chunk_ids": ", ".join(source_ids),
        "evidence_found": bool(state.get("evidence_found", False)),
        "answer_valid": bool(state.get("answer_valid", False)),
        "retry_count": int(state.get("retry_count", 0)),
        "workflow_trace": " -> ".join(state.get("trace", [])),
        "status": state.get("status", "")
    })

agentic_eval_df = pd.DataFrame(agentic_rows)
display(agentic_eval_df[["question", "agentic_answer", "source_chunk_ids", "answer_valid", "retry_count"]])

Running agentic question 1/5...
Running agentic question 2/5...
Running agentic question 3/5...
Running agentic question 4/5...
Running agentic question 5/5...


,question,agentic_answer,source_chunk_ids,answer_valid,retry_count
0,How can I track my order?,[Source ID: FAQ001_CHUNK001 | Title: Track an ...,"FAQ001_CHUNK001, FAQ012_CHUNK001, FAQ003_CHUNK...",True,0
1,What should I do if my item arrives damaged?,Contact support within 7 days of delivery. Inc...,"FAQ005_CHUNK001, POL002_CHUNK001, POL005_CHUNK...",True,0
2,How long do I have to return an item?,30 calendar days,"POL002_CHUNK001, FAQ002_CHUNK001, FAQ004_CHUNK...",True,0
3,Which payment methods are accepted?,"A payment can be declined by the bank, card is...","POL003_CHUNK001, FAQ011_CHUNK001, FAQ004_CHUNK...",True,0
4,How can I keep my account secure?,Use the password to keep your account safe.,"FAQ008_CHUNK001, FAQ011_CHUNK001, POL004_CHUNK...",True,0


## Cell 15 — Compare Phase 3 baseline and Phase 4 agentic answers

If the Phase 3 evaluation CSV exists, this joins the two outputs by question. Review answers and sources manually; extra workflow steps do not automatically mean the answers are better.

In [15]:
BASELINE_PATH = PROJECT_ROOT / "reports" / "phase3_baseline_evaluation.csv"
if BASELINE_PATH.exists():
    baseline_df = pd.read_csv(BASELINE_PATH)
    question_col = "question" if "question" in baseline_df.columns else None
    answer_candidates = ["generated_answer", "answer", "baseline_answer"]
    baseline_answer_col = next((c for c in answer_candidates if c in baseline_df.columns), None)
    if question_col and baseline_answer_col:
        comparison_df = baseline_df[[question_col, baseline_answer_col]].rename(
            columns={question_col: "question", baseline_answer_col: "baseline_answer"}
        ).merge(agentic_eval_df, on="question", how="outer")
        display(comparison_df[["question", "baseline_answer", "agentic_answer", "source_chunk_ids"]])
    else:
        print("Phase 3 file found, but its expected question/answer columns were not detected.")
        print("Available columns:", list(baseline_df.columns))
else:
    print("Phase 3 baseline file not found. Agentic results are still available.")

,question,baseline_answer,agentic_answer,source_chunk_ids
0,How can I keep my account secure?,NaN,Use the password to keep your account safe.,"FAQ008_CHUNK001, FAQ011_CHUNK001, POL004_CHUNK..."
1,How can I protect my account?,Use a password that is not a password.,NaN,NaN
2,How can I track my order?,[Source ID: FAQ001_CHUNK001 | Title: Track an ...,[Source ID: FAQ001_CHUNK001 | Title: Track an ...,"FAQ001_CHUNK001, FAQ012_CHUNK001, FAQ003_CHUNK..."
3,How long do I have to return an item?,30 calendar days,30 calendar days,"POL002_CHUNK001, FAQ002_CHUNK001, FAQ004_CHUNK..."
4,What payment methods are accepted?,"A payment can be declined by the bank, card is...",NaN,NaN
5,What should I do if my item arrives damaged?,Contact support within 7 days of delivery. Inc...,Contact support within 7 days of delivery. Inc...,"FAQ005_CHUNK001, POL002_CHUNK001, POL005_CHUNK..."
6,Which payment methods are accepted?,NaN,"A payment can be declined by the bank, card is...","POL003_CHUNK001, FAQ011_CHUNK001, FAQ004_CHUNK..."


## Cell 16 — Save the Phase 4 evaluation and trace

In [16]:
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

AGENTIC_PATH = REPORT_DIR / "phase4_agentic_evaluation.csv"
TRACE_PATH = REPORT_DIR / "phase4_workflow_traces.csv"
agentic_eval_df.to_csv(AGENTIC_PATH, index=False, encoding="utf-8")
agentic_eval_df[["question", "workflow_trace", "retry_count", "status"]].to_csv(
    TRACE_PATH, index=False, encoding="utf-8"
)

print("Saved:", AGENTIC_PATH)
print("Saved:", TRACE_PATH)

Saved: C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\reports\phase4_agentic_evaluation.csv
Saved: C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\reports\phase4_workflow_traces.csv


## Cell 17 — Final checks

A passing check confirms the graph ran and produced report files. It does not prove every generated answer is factually correct; use the manual review and compare with the knowledge base.

In [17]:
assert len(chunks_df) > 0, "No knowledge-base chunks loaded."
assert faiss_index.ntotal == len(chunks_df), "FAISS/chunk count mismatch."
assert len(agentic_eval_df) == len(evaluation_data), "Evaluation row count mismatch."
assert AGENTIC_PATH.exists(), "Agentic evaluation CSV was not saved."
assert TRACE_PATH.exists(), "Workflow trace CSV was not saved."
assert agentic_eval_df["agentic_answer"].fillna("").str.strip().ne("").all(), "At least one answer is empty."

print("Chunks loaded:", len(chunks_df))
print("FAISS vectors:", faiss_index.ntotal)
print("Agentic evaluation questions:", len(agentic_eval_df))
print("Evaluation files saved successfully.")
print("\nPHASE 4 LANGGRAPH AGENTIC RAG CHECK COMPLETE.")

Chunks loaded: 18
FAISS vectors: 18
Agentic evaluation questions: 5
Evaluation files saved successfully.

PHASE 4 LANGGRAPH AGENTIC RAG CHECK COMPLETE.
